# Final Model Evaluation

The after-departure XGBoost model was selected as the final model because it achieved the strongest validation performance and exceeded the required accuracy threshold.

In this notebook, the model is retrained using all available development data from 2019 through 2022. It is then evaluated once on the untouched 2023 test dataset.

The test set is not used for model selection, threshold adjustment, or hyperparameter tuning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    TargetEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier

In [ ]:
from pathlib import Path


def find_project_root():
    """
    Locate the project root regardless of whether the notebook
    is launched from the project root or the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Keep the data and notebooks folders inside the same project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)
df.head()

## Final Feature Set

The final model predicts arrival delay immediately after departure.

Therefore, the feature set includes actual departure information:

- `DEP_TIME`
- `DEP_DELAY`

Arrival-related variables, post-arrival variables, and delay-cause variables remain excluded to prevent target leakage.

In [ ]:
selected_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST",

    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",

    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "QUARTER",

    "DEP_HOUR",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

target = "IS_DELAYED"

print("Number of selected features:", len(selected_features))

## Final Chronological Split

The final training dataset combines observations from 2019 through 2022.

The 2023 observations remain the untouched test set and are used only for the final model evaluation.

In [ ]:
final_train_df = df[
    df["YEAR"] <= 2022
].copy()

test_df = df[
    df["YEAR"] == 2023
].copy()

In [ ]:
X_final_train = final_train_df[selected_features]
y_final_train = final_train_df[target]

X_test = test_df[selected_features]
y_test = test_df[target]

In [ ]:
final_split_summary = pd.DataFrame({
    "Dataset": [
        "Final Training",
        "Final Test"
    ],
    "Years": [
        "2019–2022",
        "2023"
    ],
    "Rows": [
        len(X_final_train),
        len(X_test)
    ],
    "Delayed Flights": [
        y_final_train.sum(),
        y_test.sum()
    ],
    "Delay Rate": [
        y_final_train.mean(),
        y_test.mean()
    ]
})

final_split_summary

In [ ]:
numerical_features = [
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "YEAR",
    "MONTH",
    "DAY",
    "QUARTER",
    "DEP_HOUR",
    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"
]

one_hot_features = [
    "DAY_OF_WEEK",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY"
]

target_encode_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

one_hot_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

target_transformer = Pipeline(
    steps=[
        (
            "target_encoder",
            TargetEncoder(
                target_type="binary",
                random_state=42
            )
        )
    ]
)

In [ ]:
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "onehot",
            one_hot_transformer,
            one_hot_features
        ),
        (
            "target",
            target_transformer,
            target_encode_features
        )
    ],
    remainder="drop"
)

In [ ]:
final_xgb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            final_preprocessor
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

final_xgb_pipeline

In [ ]:
print("Training the final XGBoost model...")

final_xgb_pipeline.fit(
    X_final_train,
    y_final_train
)

print("Final model training completed.")

In [ ]:
y_test_pred = final_xgb_pipeline.predict(
    X_test
)

y_test_prob = final_xgb_pipeline.predict_proba(
    X_test
)[:, 1]

In [ ]:
final_test_results = {
    "Model": "Final After-Departure XGBoost",
    "Accuracy": accuracy_score(
        y_test,
        y_test_pred
    ),
    "Precision": precision_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        y_test_pred,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_test,
        y_test_prob
    )
}

pd.DataFrame(
    [final_test_results]
)

In [ ]:
print("Final Test Classification Report\n")

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=[
            "On Time",
            "Delayed"
        ],
        zero_division=0
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=[
        "On Time",
        "Delayed"
    ]
)

plt.title(
    "Final XGBoost Test Confusion Matrix"
)

plt.show()

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_test_prob
)

plt.title(
    "Final XGBoost ROC Curve"
)

plt.show()

In [ ]:
from pathlib import Path

models_directory = Path("../models")
models_directory.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
model_path = (
    models_directory
    / "final_after_departure_xgboost_pipeline.joblib"
)

joblib.dump(
    final_xgb_pipeline,
    model_path
)

print(
    f"Final model saved to: {models_directory}"
)

In [ ]:
results_path = (
    models_directory
    / "final_test_metrics.csv"
)

pd.DataFrame(
    [final_test_results]
).to_csv(
    results_path,
    index=False
)

print(
    f"Final metrics saved to: {results_path}"
)


## Final Test Results Interpretation

The final after-departure XGBoost model was evaluated on the untouched 2023 test dataset.

The model achieved approximately 93% test accuracy, exceeding the required minimum accuracy of 85%. It also achieved approximately 92% Precision, 73% Recall, and an F1-score of 81% for delayed flights.

The high Precision indicates that when the model predicts a flight will be delayed, the prediction is usually correct. The Recall indicates that the model successfully identifies approximately 73% of all delayed flights.

The final test performance is very similar to the 2022 validation performance. This consistency suggests that the selected model generalizes effectively to unseen future data and does not show clear evidence of overfitting.

The model performs particularly well because actual departure information, especially `DEP_DELAY`, is available at the prediction moment. Therefore, the final system should be interpreted as an after-departure arrival-delay prediction model.

In [ ]:
validation_test_comparison = pd.DataFrame([
    {
        "Dataset": "Validation",
        "Year": 2022,
        "Accuracy": 0.931529,
        "Precision": 0.922773,
        "Recall": 0.723683,
        "F1": 0.811191,
        "ROC_AUC": 0.936866
    },
    {
        "Dataset": "Test",
        "Year": 2023,
        "Accuracy": final_test_results["Accuracy"],
        "Precision": final_test_results["Precision"],
        "Recall": final_test_results["Recall"],
        "F1": final_test_results["F1"],
        "ROC_AUC": final_test_results["ROC_AUC"]
    }
])

validation_test_comparison

## Final Conclusion

This project developed and evaluated machine learning models for predicting flight arrival delays.

The initial models used only information available before departure. Logistic Regression, Random Forest, and XGBoost achieved validation accuracy of approximately 77% to 80%. Hyperparameter tuning did not produce a meaningful improvement because the available pre-departure features contained limited predictive information.

The prediction scenario was subsequently changed to predict arrival delays immediately after departure. Actual departure information, including `DEP_TIME` and `DEP_DELAY`, was added to the feature set.

The selected after-departure XGBoost model achieved approximately 93.15% accuracy on the 2022 validation dataset. After model selection, the final model was retrained using data from 2019 through 2022 and evaluated once on the untouched 2023 test dataset.

The final model achieved approximately 93% test accuracy, 92% Precision, 73% Recall, and an F1-score of 81% for delayed flights. The similarity between validation and test performance indicates that the model generalizes effectively to unseen data.

The project accuracy requirement of at least 85% was successfully achieved. However, the final model should be described specifically as an after-departure arrival-delay prediction system because it relies on actual departure information.

## Save Deployment Metadata

The Streamlit application requires valid airline and airport options. These values are extracted from the final training dataset and saved separately so the application does not need to load the complete flight dataset.

In [ ]:
deployment_metadata = {
    "airlines": sorted(
        X_final_train["AIRLINE"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "origins": sorted(
        X_final_train["ORIGIN"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "destinations": sorted(
        X_final_train["DEST"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    ),

    "defaults": {
        "CRS_ELAPSED_TIME": float(
            X_final_train["CRS_ELAPSED_TIME"].median()
        ),

        "DISTANCE": float(
            X_final_train["DISTANCE"].median()
        )
    }
}

In [ ]:
metadata_path = (
    models_directory
    / "deployment_metadata.joblib"
)

joblib.dump(
    deployment_metadata,
    metadata_path
)

print(f"Deployment metadata saved to: {metadata_path}")

In [ ]:
print(
    "Airlines:",
    len(deployment_metadata["airlines"])
)

print(
    "Origin airports:",
    len(deployment_metadata["origins"])
)

print(
    "Destination airports:",
    len(deployment_metadata["destinations"])
)

print(
    "Default values:",
    deployment_metadata["defaults"]
)